# Phase 01.03 — LoRA r16 training (F1)

Fine-tunes only LoRA adapters on the frozen Phase 01 training split. Targets are `q_proj/k_proj/v_proj/o_proj`, rank is fixed at 16, labels supervise only assistant answer tokens using token-prefix masking, and every visual tile receives one `image_flag`.


In [1]:
import os, sys
from pathlib import Path

PROJECT_ROOT = Path("/workspace/RoadBuddy")
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

os.environ.setdefault("CC", "/usr/bin/gcc")
os.environ.setdefault("CXX", "/usr/bin/g++")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

from roadbuddy_common import *

os.chdir(PROJECT_ROOT)
seed_everything(SEED)
ensure_dirs()
print("Project root:", PROJECT_ROOT)
print("Model revision:", MODEL_REVISION)


Project root: /workspace/RoadBuddy
Model revision: b98f263eab246eb5269ade64edbdca8a887dc44d


## 1. Training configuration

Start with `DEBUG_LIMIT=20` and `MAX_STEPS=2`. After the smoke run passes, set both to `None` for the full run.


In [2]:
from torch.utils.data import DataLoader
from peft import LoraConfig, get_peft_model

DEBUG_LIMIT = 20
MAX_STEPS = 2
EPOCHS = 1
BATCH_SIZE = 1
GRADIENT_ACCUMULATION = 16
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 0.01
MAX_GRAD_NORM = 1.0
NUM_WORKERS = 0

TRAIN_CSV = PATHS.phase1_split / "train.csv"
assert TRAIN_CSV.is_file(), "Run notebook 01 first"
train_df = pd.read_csv(TRAIN_CSV)
print("Selected rows:", min(len(train_df), DEBUG_LIMIT) if DEBUG_LIMIT else len(train_df))


/root/venvs/roadbuddy-rtx3090-py310/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Selected rows: 20


## 2. Load pinned model and attach LoRA


In [3]:
model, tokenizer = load_model_and_tokenizer(training=True)
model.img_context_token_id = tokenizer.convert_tokens_to_ids(IMG_CONTEXT_TOKEN)

for parameter in model.parameters():
    parameter.requires_grad = False

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
assert trainable > 0
assert all((p.requires_grad == ("lora_" in name)) for name, p in model.named_parameters()), "Only LoRA parameters may be trainable"


/root/venvs/roadbuddy-rtx3090-py310/lib/python3.10/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


FlashAttention2 is not installed.


trainable params: 2,162,688 || all params: 940,355,712 || trainable%: 0.2300


## 3. Dataset and alignment smoke check

This cell materializes one sample and checks visual-token alignment, supervised token count, and per-tile flags before training.


In [4]:
dataset = RoadBuddySFTDataset(train_df, model, tokenizer, limit=DEBUG_LIMIT)
collator = make_collator(tokenizer)
sample = dataset[0]
print({
    "sample_id": sample["sample_id"],
    "tiles": len(sample["pixel_values"]),
    "sequence_tokens": len(sample["input_ids"]),
    "supervised_tokens": int((sample["labels"] != IGNORE_INDEX).sum()),
    "image_flags": len(sample["image_flags"]),
    "num_image_token": int(model.num_image_token),
})
assert len(sample["pixel_values"]) == len(sample["image_flags"])


{'sample_id': 'train_0001', 'tiles': 7, 'sequence_tokens': 1919, 'supervised_tokens': 2, 'image_flags': 7, 'num_image_token': 256}


## 4. Correctness-first training loop


In [5]:
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, collate_fn=collator, generator=torch.Generator().manual_seed(SEED))
optimizer = torch.optim.AdamW((p for p in model.parameters() if p.requires_grad), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

history = []
global_step = 0
optimizer.zero_grad(set_to_none=True)
torch.cuda.reset_peak_memory_stats()

for epoch in range(EPOCHS):
    for micro_step, batch in enumerate(loader, start=1):
        sample_ids = batch.pop("sample_ids")
        batch = {key: value.cuda(non_blocking=True) for key, value in batch.items()}
        assert batch["image_flags"].numel() == batch["pixel_values"].shape[0]
        with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
            output = model(**batch)
            loss = output.loss / GRADIENT_ACCUMULATION
        loss.backward()
        should_step = micro_step % GRADIENT_ACCUMULATION == 0 or micro_step == len(loader)
        if should_step:
            torch.nn.utils.clip_grad_norm_((p for p in model.parameters() if p.requires_grad), MAX_GRAD_NORM)
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)
            global_step += 1
            record = {"epoch": epoch + 1, "step": global_step, "loss": float(loss.item() * GRADIENT_ACCUMULATION), "sample_ids": sample_ids}
            history.append(record)
            print(record)
            if MAX_STEPS is not None and global_step >= MAX_STEPS:
                break
    if MAX_STEPS is not None and global_step >= MAX_STEPS:
        break

peak_gib = torch.cuda.max_memory_allocated() / 1024**3
print(f"Peak allocated VRAM: {peak_gib:.2f} GiB")


/root/venvs/roadbuddy-rtx3090-py310/lib/python3.10/site-packages/torch/_dynamo/eval_frame.py:1446: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/root/venvs/roadbuddy-rtx3090-py310/lib/python3.10/site-packages/torch/utils/checkpoint.py:238: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)


/root/venvs/roadbuddy-rtx3090-py310/lib/python3.10/site-packages/torch/_dynamo/eval_frame.py:1446: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/root/venvs/roadbuddy-rtx3090-py310/lib/python3.10/site-packages/torch/utils/checkpoint.py:238: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)


{'epoch': 1, 'step': 1, 'loss': 1.9296875, 'sample_ids': ['train_0015']}


{'epoch': 1, 'step': 2, 'loss': 0.6562516093254089, 'sample_ids': ['train_0019']}
Peak allocated VRAM: 14.80 GiB


## 5. Save adapter and provenance

Debug and full runs use different directories, preventing accidental overwrite.


In [6]:
run_name = "debug20" if DEBUG_LIMIT else "full"
out_dir = PATHS.lora / run_name
adapter_dir = out_dir / "checkpoints" / "adapter_final"
adapter_dir.mkdir(parents=True, exist_ok=True)
model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)
pd.DataFrame(history).to_csv(out_dir / "train_history.csv", index=False)
save_json(out_dir / "train_metrics.json", {"optimizer_steps": global_step, "peak_vram_gib": peak_gib, "last_loss": history[-1]["loss"] if history else None})
save_json(out_dir / "config.json", {
    "model_id": MODEL_ID, "revision": MODEL_REVISION, "seed": SEED, "frames": 1,
    "lora_rank": 16, "lora_alpha": 32, "lora_dropout": 0.05,
    "targets": ["q_proj", "k_proj", "v_proj", "o_proj"],
    "label_masking": "tokenized_native_template_prefix", "template": "Hermes-2",
    "num_image_token": int(model.num_image_token), "image_flags": "one_per_visual_tile",
    "flash_attention": False, "debug_limit": DEBUG_LIMIT, "max_steps": MAX_STEPS,
    "epochs": EPOCHS, "batch_size": BATCH_SIZE, "gradient_accumulation": GRADIENT_ACCUMULATION,
    "learning_rate": LEARNING_RATE,
})
print("Saved adapter:", adapter_dir)


Saved adapter: /workspace/RoadBuddy/outputs/phase01/lora_r16_f1/debug20/checkpoints/adapter_final


## Nhận xét sau lần chạy Phase 01.03

**Phạm vi:** smoke/debug training với 20 rows, LoRA rank 16 và tối đa 2 optimizer steps. Đây là kiểm tra correctness của pipeline, không phải một lần fine-tune hội tụ.

### Cấu hình và tài nguyên

- LoRA target: `q_proj`, `k_proj`, `v_proj`, `o_proj`; rank 16, alpha 32, dropout 0.05.
- Trainable parameters: 2.162.688 / 940.355.712, tương đương 0,23%; chỉ tham số LoRA được mở gradient.
- Batch size 1, gradient accumulation 16, learning rate `1e-4`, BF16.
- Sample alignment probe: 7 tiles, khoảng 1.919 tokens, 2 supervised answer tokens; số `image_flags` khớp số tiles.
- Loss được log: 1,9297 ở step 1 và 0,6563 ở step 2.
- Peak allocated VRAM: 14,80 GiB, nằm trong giới hạn RTX 3090 24 GiB.

### Nhận định

Token/image alignment, label masking, forward/backward, optimizer step và save/reload adapter đều hoạt động. Tokenizer max length đã được đồng bộ ở 4.096, thấp hơn context window 32.768 của language model và đủ cho mẫu 7 tiles hiện tại.

Không nên diễn giải hai giá trị loss là bằng chứng hội tụ: chỉ có 2 optimizer steps và step cuối tích lũy ít micro-batch hơn step đầu. Ngoài ra `sample_ids` trong history hiện chỉ ghi micro-batch cuối tại thời điểm optimizer step, chưa ghi toàn bộ các sample đã đóng góp vào gradient accumulation.

### Khuyến nghị trước full training

- Đặt `DEBUG_LIMIT=None` và chọn `MAX_STEPS`/epochs theo kế hoạch đầy đủ.
- Sửa provenance logging để lưu toàn bộ accumulated sample IDs.
- Chuẩn hóa loss logging cho partial accumulation cuối epoch.
- Theo dõi validation metric theo checkpoint và giữ seed/split hiện tại để so sánh công bằng.